# Comparative Synthesis: Payoff-Based Learning & Extremum Seeking under Multi-Task Bayesian Optimization

This notebook synthesizes the empirical evaluations, sensitivity analyses, and convergence dynamics comparing two continuous-action, zeroth-order learning paradigms across a 100-game ensemble of strongly monotone quadratic games:
1. **Zero-Order Gradient Play (`ZOGrandientPlay`)**: Stochastic Gaussian exploration with regularized interior projections (Tatarenko & Kamgarpour, 2019/2022).
2. **Continuous-Time Extremum Seeking (`NESeeking5D`)**: Sinusoidal perturbation dynamics with boundary projections and demodulation filters (Frihauf et al., 2012).

---

### Core Methodological Frameworks
- **Cost Minimization Formulation ($C_i = -J_i$)**: All game utilities are presented in the cost minimization framework. Regret $R_i(t) \ge 0$ remains mathematically invariant.
- **Logarithmic Regret Metrics**: Evaluated as $\log_{10}(\text{Regret}_t)$ and $\log_{10}(\text{Total Regret})$.
- **Visual Styling & Palette**:
  - **MTBO Tailored Optima**: RIS-Grün (`#009D81` / RGB `[0, 157, 120]`).
  - **Universal Consensus Default**: Warm Orange (`#FF7F0E`).
  - **Ground Truth Nash Equilibrium**: Dashed Black (`k--`).
  - **Optuna Artifacts**: Strictly omitted.
- **Sensitivity & Convergence Probes**:
  - **Sobol Variance Decomposition**: $N = 256$ Sobol samples ($2^8$ dyadic net) evaluated over $T = 5,000$ steps in parallel across 5 representative games.
  - **100-Game Convergence Probe**: Evaluated under the **Universal Default Hyperparameters** to eliminate hyperparameter variance as an uncontrolled confounder, checking stopping within $\epsilon_{\text{tol}} = 0.05$ of the true Nash equilibrium up to $T_{\max} = 100,000$ steps.

All figures, metrics tables, and CSV outputs are co-located in `data/08_final_results/`.

In [ ]:
import os
import time
from concurrent.futures import ProcessPoolExecutor
import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import qmc
import matplotlib.pyplot as plt

from payoffnash import (
    Game,
    QuadraticGame,
    generate_quadratic_games,
    ZOGrandientPlay,
    NESeeking5D,
    EstimateHistory,
    quadratic_game_regret_cost,
    eval_zo_sobol_worker,
    eval_ne_sobol_worker,
    run_game_sobol,
    run_convergence_worker,
)

# 1. Styling Constants
RIS_GREEN = "#009D81"         # MTBO Tailored Optima
ORANGE_DEFAULT = "#FF7F0E"    # Universal Consensus Default
BLACK_NE = "#000000"          # Ground Truth Nash Equilibrium

# Monochromatic RIS-Grün Palette for Sobol Bar Charts
# Multi-Color Palette for Sobol: Game 11 in RIS-Grün, distinct colors for others
SOBOL_PALETTE = ["#009D81", "#2E5B88", "#D95F02", "#7570B3", "#E7298A"]

plt.rcParams["font.sans-serif"] = "DejaVu Sans"
plt.rcParams["axes.edgecolor"] = "#333333"
plt.rcParams["axes.linewidth"] = 0.8

# 2. Output Directory Co-location (Rule 4)
data_dir = "data/08_final_results"
os.makedirs(data_dir, exist_ok=True)
print(f"All artifacts and plots will be saved to: {data_dir}")

In [ ]:
# 1. Generate the standardized 100-game ensemble (Seed: 14062003)
games = generate_quadratic_games(
    ensemble_size=100,
    N=2,
    m=1,
    alpha=0.05,
    enforce_interior=True,
    seed=14062003,
)

# 2. Load MTBO batch results and extract tailored and universal default parameters
def load_all_parameters():
    saa_dir = "data/06_saa_tuned_100_games"
    zo_tailored = {}
    ne_tailored = {}
    zo_default = None
    ne_default = None

    g0_bounds = np.array([[iv[1] - iv[0] for iv in s] for s in games[0].action_sets], dtype=float)
    g0_min_w = float(np.min(g0_bounds))
    g0_w0 = float(g0_bounds[0, 0])
    g0_w1 = float(g0_bounds[1, 0])

    for b in range(4):
        df_zo = pd.read_csv(os.path.join(saa_dir, f"zo_batch_{b}_results.csv"))
        for _, row in df_zo[df_zo["is_default"] == False].iterrows():
            gid = b * 25 + int(str(row["game_id"]).replace("game_", ""))
            zo_tailored[gid] = {"v": float(row["v"]), "a": float(row["a"]), "epsilon": float(row["epsilon"])}
        if zo_default is None:
            def_row = df_zo[df_zo["is_default"] == True].iloc[0]
            # Unscale Game 0 default parameter to recover dimensionless parameters:
            zo_default = {"v": float(def_row["v"]), "a": float(def_row["a"]) / g0_min_w, "epsilon": float(def_row["epsilon"])}

        df_ne = pd.read_csv(os.path.join(saa_dir, f"ne_batch_{b}_results.csv"))
        for _, row in df_ne[df_ne["is_default"] == False].iterrows():
            gid = b * 25 + int(str(row["game_id"]).replace("game_", ""))
            ne_tailored[gid] = {
                "k0": float(row["k[0,0]"]),
                "k1": float(row["k[1,0]"]),
                "a0": float(row["a[0,0]"]),
                "a1": float(row["a[1,0]"]),
                "omega1": float(row["omega1"]),
                "dt": float(row["dt"]),
            }
        if ne_default is None:
            def_row_ne = df_ne[df_ne["is_default"] == True].iloc[0]
            # Unscale Game 0 default parameters to recover dimensionless parameters:
            ne_default = {
                "k0": float(def_row_ne["k[0,0]"]),
                "k1": float(def_row_ne["k[1,0]"]),
                "a0": float(def_row_ne["a[0,0]"]) / g0_w0,
                "a1": float(def_row_ne["a[1,0]"]) / g0_w1,
                "omega1": float(def_row_ne["omega1"]),
                "dt": float(def_row_ne["dt"]),
            }

    return zo_tailored, zo_default, ne_tailored, ne_default

zo_tailored, zo_default, ne_tailored, ne_default = load_all_parameters()
ensemble_metrics_df = pd.read_csv("data/06_saa_tuned_100_games/ensemble_100_games_metrics.csv")

print("Universal Consensus Defaults (Dimensionless Perturbation Scale):")
print("  ZOGP Default:      ", zo_default)
print("  NESeeking5D Default:", ne_default)
print(f"Loaded tailored parameters for {len(zo_tailored)} ZOGP games and {len(ne_tailored)} NESeeking games.")

## 1. The 5 Representative Games (Cost Minimization Formulation)

To capture distinct structural challenges of multi-agent learning, 5 representative games were selected from the 100-game ensemble based on extreme geometric criteria:

In the cost minimization framework, each player $i$ minimizes $C_i(u) = -J_i(u)$:
$$
C_i(u) = \frac{1}{2} u_i^T Q_{ii}^{\text{cost}} u_i + u_i^T \sum_{j \neq i} Q_{ij}^{\text{cost}} u_j + (r_i^{\text{cost}})^T u_i + c_i^{\text{cost}}
$$
where $Q^{\text{cost}} = -H$, $r^{\text{cost}} = -d$, and $c^{\text{cost}} = -c$.

### Exact Mathematical Cost Specifications:
- **Game 11 (Extreme Ill-Conditioning, $\kappa = 15.16$):**
  - Action Bounds: $\mathcal{A}_1 = [-0.717, 0.452]$, $\mathcal{A}_2 = [-0.300, 0.501]$.
  - Nash Equilibrium: $u^* = [0.2890, 0.0735]$.
  - $Q^{\text{cost}} = \begin{bmatrix} 0.1254 & 0.9838 \\ -0.5338 & 0.8027 \end{bmatrix}$, $r^{\text{cost}} = \begin{bmatrix} -0.1086 \\ 0.0952 \end{bmatrix}$, $c^{\text{cost}} = \begin{bmatrix} -0.6779 \\ -0.1699 \end{bmatrix}$.

- **Game 70 (Well-Behaved / Isotropic, $\kappa = 1.05$):**
  - Action Bounds: $\mathcal{A}_1 = [-0.535, 0.364]$, $\mathcal{A}_2 = [-0.884, 0.672]$.
  - Nash Equilibrium: $u^* = [0.3248, 0.4164]$.
  - $Q^{\text{cost}} = \begin{bmatrix} 0.8615 & -0.9608 \\ 0.9178 & 0.8776 \end{bmatrix}$, $r^{\text{cost}} = \begin{bmatrix} 0.1203 \\ -0.6636 \end{bmatrix}$, $c^{\text{cost}} = \begin{bmatrix} 0.9128 \\ 0.2988 \end{bmatrix}$.

- **Game 97 (Narrow Action Bounds, $\text{vol} = 0.0818$):**
  - Action Bounds: $\mathcal{A}_1 = [-0.270, 0.502]$, $\mathcal{A}_2 = [-0.040, 0.066]$.
  - Nash Equilibrium: $u^* = [-0.0507, 0.0480]$.
  - $Q^{\text{cost}} = \begin{bmatrix} 0.9532 & -0.4457 \\ -0.0769 & 0.4117 \end{bmatrix}$, $r^{\text{cost}} = \begin{bmatrix} 0.0697 \\ -0.0237 \end{bmatrix}$, $c^{\text{cost}} = \begin{bmatrix} -0.5819 \\ 0.2191 \end{bmatrix}$.

- **Game 40 (Highly Asymmetric Coupling, Skewness = $2.781$):**
  - Action Bounds: $\mathcal{A}_1 = [-0.824, 0.599]$, $\mathcal{A}_2 = [-0.804, 0.894]$.
  - Nash Equilibrium: $u^* = [0.1650, 0.2749]$.
  - $Q^{\text{cost}} = \begin{bmatrix} 0.2913 & 0.9797 \\ -0.9868 & 0.5180 \end{bmatrix}$, $r^{\text{cost}} = \begin{bmatrix} -0.3174 \\ 0.0204 \end{bmatrix}$, $c^{\text{cost}} = \begin{bmatrix} 0.8394 \\ -0.4772 \end{bmatrix}$.

- **Game 17 (Boundary-Adjacent Equilibrium, Dist to Boundary = $0.006$):**
  - Action Bounds: $\mathcal{A}_1 = [-0.219, 0.535]$, $\mathcal{A}_2 = [-0.668, 0.643]$.
  - Nash Equilibrium: $u^* = [0.1908, 0.6364]$.
  - $Q^{\text{cost}} = \begin{bmatrix} 0.4567 & 0.0817 \\ 0.1824 & 0.9061 \end{bmatrix}$, $r^{\text{cost}} = \begin{bmatrix} -0.1392 \\ -0.6114 \end{bmatrix}$, $c^{\text{cost}} = \begin{bmatrix} -0.2630 \\ 0.4607 \end{bmatrix}$.

In [ ]:
# 2-Way Comparative Trajectory Visualizations (Tailored in RIS-Grün vs. Universal Default in Orange)
rep_games_info = [
    (11, "Extreme Ill-Conditioned (kappa=15.16)"),
    (70, "Well-Behaved / Isotropic (kappa=1.05)"),
    (97, "Narrow Action Bounds (vol=0.0818)"),
    (40, "Highly Asymmetric Coupling (skew=2.781)"),
    (17, "Boundary-Adjacent Equilibrium (dist=0.006)"),
]

num_sim_steps = 20000

for gid, category in rep_games_info:
    g = games[gid]
    u_star = g.nash_equilibrium.ravel()
    bounds = np.array([[iv[1] - iv[0] for iv in s] for s in g.action_sets], dtype=float)
    min_w = float(np.min(bounds))
    w0, w1 = float(bounds[0, 0]), float(bounds[1, 0])
    lb0, ub0 = g.action_sets[0][0][0], g.action_sets[0][0][1]
    lb1, ub1 = g.action_sets[1][0][0], g.action_sets[1][0][1]
    mid_actions = np.array([[0.5 * (iv[0] + iv[1]) for iv in s] for s in g.action_sets], dtype=float)

    # ------------------ ZOGrandientPlay ------------------
    zo_p_tail = zo_tailored[gid]
    np.random.seed(14062003)
    est_zo_tail = ZOGrandientPlay(g, v=zo_p_tail["v"], a=zo_p_tail["a"], epsilon=zo_p_tail["epsilon"])
    res_zo_tail = est_zo_tail.estimate(mid_actions, num_steps=num_sim_steps)
    loss_zo_tail = quadratic_game_regret_cost(g, res_zo_tail)

    np.random.seed(14062003)
    est_zo_def = ZOGrandientPlay(g, v=zo_default["v"], a=zo_default["a"] * min_w, epsilon=zo_default["epsilon"])
    res_zo_def = est_zo_def.estimate(mid_actions, num_steps=num_sim_steps)
    loss_zo_def = quadratic_game_regret_cost(g, res_zo_def)

    t_steps = np.arange(len(res_zo_tail.action_history))

    # ZOGP 4-Panel Visualization (2x2: Player 1 Action, Player 2 Action, Player 1 State, Player 2 State)
    fig, axes = plt.subplots(2, 2, figsize=(14, 9.5))
    fig.suptitle(f"ZOGrandientPlay: Game {gid} ({category})\nTailored (LogCost={loss_zo_tail:.3f}) vs. Default (LogCost={loss_zo_def:.3f}) | NE: [{u_star[0]:.3f}, {u_star[1]:.3f}]", fontsize=13, fontweight="bold")

    # (a) Player 1 Action History
    axes[0, 0].plot(t_steps, res_zo_tail.action_history[:, 0], color=RIS_GREEN, lw=1.0, alpha=0.85, label=f"Tailored Action (LogCost={loss_zo_tail:.3f})")
    axes[0, 0].plot(t_steps, res_zo_def.action_history[:, 0], color=ORANGE_DEFAULT, lw=0.8, alpha=0.75, label=f"Default Action (LogCost={loss_zo_def:.3f})")
    axes[0, 0].axhline(u_star[0], color=BLACK_NE, linestyle="--", lw=1.5, label=f"True NE $u_1^* = {u_star[0]:.3f}$")
    axes[0, 0].axhline(lb0, color="gray", linestyle=":", lw=1.2, alpha=0.6, label=f"Bounds [{lb0:.2f}, {ub0:.2f}]")
    axes[0, 0].axhline(ub0, color="gray", linestyle=":", lw=1.2, alpha=0.6)
    axes[0, 0].set_title(r"Player 1: Action History $u_1(t)$")
    axes[0, 0].set_xlabel("Iteration Step $t$")
    axes[0, 0].set_ylabel(r"Action $u_1$")
    axes[0, 0].grid(True, alpha=0.3)
    axes[0, 0].legend(loc="best", fontsize=8.5)

    # (b) Player 2 Action History
    axes[0, 1].plot(t_steps, res_zo_tail.action_history[:, 1], color=RIS_GREEN, lw=1.0, alpha=0.85, label="Tailored Action")
    axes[0, 1].plot(t_steps, res_zo_def.action_history[:, 1], color=ORANGE_DEFAULT, lw=0.8, alpha=0.75, label="Default Action")
    axes[0, 1].axhline(u_star[1], color=BLACK_NE, linestyle="--", lw=1.5, label=f"True NE $u_2^* = {u_star[1]:.3f}$")
    axes[0, 1].axhline(lb1, color="gray", linestyle=":", lw=1.2, alpha=0.6, label=f"Bounds [{lb1:.2f}, {ub1:.2f}]")
    axes[0, 1].axhline(ub1, color="gray", linestyle=":", lw=1.2, alpha=0.6)
    axes[0, 1].set_title(r"Player 2: Action History $u_2(t)$")
    axes[0, 1].set_xlabel("Iteration Step $t$")
    axes[0, 1].set_ylabel(r"Action $u_2$")
    axes[0, 1].grid(True, alpha=0.3)
    axes[0, 1].legend(loc="best", fontsize=8.5)

    # (c) Player 1 State History
    axes[1, 0].plot(t_steps, res_zo_tail.state_history[:, 0], color=RIS_GREEN, lw=1.8, label=r"Tailored State $\mu_1(t)$")
    axes[1, 0].plot(t_steps, res_zo_def.state_history[:, 0], color=ORANGE_DEFAULT, lw=1.5, alpha=0.85, label=r"Default State $\mu_1(t)$")
    axes[1, 0].axhline(u_star[0], color=BLACK_NE, linestyle="--", lw=1.5, label=f"True NE $u_1^* = {u_star[0]:.3f}$")
    axes[1, 0].axhline(lb0, color="gray", linestyle=":", lw=1.2, alpha=0.6)
    axes[1, 0].axhline(ub0, color="gray", linestyle=":", lw=1.2, alpha=0.6)
    axes[1, 0].set_title(r"Player 1: Internal State History $\mu_1(t)$")
    axes[1, 0].set_xlabel("Iteration Step $t$")
    axes[1, 0].set_ylabel(r"State $\mu_1$")
    axes[1, 0].grid(True, alpha=0.3)
    axes[1, 0].legend(loc="best", fontsize=8.5)

    # (d) Player 2 State History
    axes[1, 1].plot(t_steps, res_zo_tail.state_history[:, 1], color=RIS_GREEN, lw=1.8, label=r"Tailored State $\mu_2(t)$")
    axes[1, 1].plot(t_steps, res_zo_def.state_history[:, 1], color=ORANGE_DEFAULT, lw=1.5, alpha=0.85, label=r"Default State $\mu_2(t)$")
    axes[1, 1].axhline(u_star[1], color=BLACK_NE, linestyle="--", lw=1.5, label=f"True NE $u_2^* = {u_star[1]:.3f}$")
    axes[1, 1].axhline(lb1, color="gray", linestyle=":", lw=1.2, alpha=0.6)
    axes[1, 1].axhline(ub1, color="gray", linestyle=":", lw=1.2, alpha=0.6)
    axes[1, 1].set_title(r"Player 2: Internal State History $\mu_2(t)$")
    axes[1, 1].set_xlabel("Iteration Step $t$")
    axes[1, 1].set_ylabel(r"State $\mu_2$")
    axes[1, 1].grid(True, alpha=0.3)
    axes[1, 1].legend(loc="best", fontsize=8.5)

    plt.tight_layout()
    fig_path = os.path.join(data_dir, f"rep_game_{gid}_zogp.png")
    plt.savefig(fig_path, dpi=300, bbox_inches="tight")
    plt.close(fig)

    # ------------------ NESeeking5D ------------------
    ne_p_tail = ne_tailored[gid]
    est_ne_tail = NESeeking5D(
        g,
        k=np.array([[ne_p_tail["k0"]], [ne_p_tail["k1"]]]),
        a=np.array([[ne_p_tail["a0"]], [ne_p_tail["a1"]]]),
        omega1=ne_p_tail["omega1"],
        dt=ne_p_tail["dt"],
    )
    res_ne_tail = est_ne_tail.estimate(mid_actions, num_steps=num_sim_steps)
    loss_ne_tail = quadratic_game_regret_cost(g, res_ne_tail)

    est_ne_def = NESeeking5D(
        g,
        k=np.array([[ne_default["k0"]], [ne_default["k1"]]]),
        a=np.array([[ne_default["a0"] * w0], [ne_default["a1"] * w1]]),
        omega1=ne_default["omega1"],
        dt=ne_default["dt"],
    )
    res_ne_def = est_ne_def.estimate(mid_actions, num_steps=num_sim_steps)
    loss_ne_def = quadratic_game_regret_cost(g, res_ne_def)

    # NESeeking 4-Panel Visualization (2x2: Player 1 Action, Player 2 Action, Player 1 State, Player 2 State)
    fig, axes = plt.subplots(2, 2, figsize=(14, 9.5))
    fig.suptitle(f"NESeeking5D: Game {gid} ({category})\nTailored (LogCost={loss_ne_tail:.3f}) vs. Default (LogCost={loss_ne_def:.3f}) | NE: [{u_star[0]:.3f}, {u_star[1]:.3f}]", fontsize=13, fontweight="bold")

    # (a) Player 1 Action History
    axes[0, 0].plot(t_steps, res_ne_tail.action_history[:, 0], color=RIS_GREEN, lw=0.9, alpha=0.85, label=f"Tailored Action (LogCost={loss_ne_tail:.3f})")
    axes[0, 0].plot(t_steps, res_ne_def.action_history[:, 0], color=ORANGE_DEFAULT, lw=0.7, alpha=0.75, label=f"Default Action (LogCost={loss_ne_def:.3f})")
    axes[0, 0].axhline(u_star[0], color=BLACK_NE, linestyle="--", lw=1.5, label=f"True NE $u_1^* = {u_star[0]:.3f}$")
    axes[0, 0].axhline(lb0, color="gray", linestyle=":", lw=1.2, alpha=0.6, label=f"Bounds [{lb0:.2f}, {ub0:.2f}]")
    axes[0, 0].axhline(ub0, color="gray", linestyle=":", lw=1.2, alpha=0.6)
    axes[0, 0].set_title(r"Player 1: Action History $u_1(t)$")
    axes[0, 0].set_xlabel("Iteration Step $t$")
    axes[0, 0].set_ylabel(r"Action $u_1$")
    axes[0, 0].grid(True, alpha=0.3)
    axes[0, 0].legend(loc="best", fontsize=8.5)

    # (b) Player 2 Action History
    axes[0, 1].plot(t_steps, res_ne_tail.action_history[:, 1], color=RIS_GREEN, lw=0.9, alpha=0.85, label="Tailored Action")
    axes[0, 1].plot(t_steps, res_ne_def.action_history[:, 1], color=ORANGE_DEFAULT, lw=0.7, alpha=0.75, label="Default Action")
    axes[0, 1].axhline(u_star[1], color=BLACK_NE, linestyle="--", lw=1.5, label=f"True NE $u_2^* = {u_star[1]:.3f}$")
    axes[0, 1].axhline(lb1, color="gray", linestyle=":", lw=1.2, alpha=0.6, label=f"Bounds [{lb1:.2f}, {ub1:.2f}]")
    axes[0, 1].axhline(ub1, color="gray", linestyle=":", lw=1.2, alpha=0.6)
    axes[0, 1].set_title(r"Player 2: Action History $u_2(t)$")
    axes[0, 1].set_xlabel("Iteration Step $t$")
    axes[0, 1].set_ylabel(r"Action $u_2$")
    axes[0, 1].grid(True, alpha=0.3)
    axes[0, 1].legend(loc="best", fontsize=8.5)

    # (c) Player 1 Demodulated State History
    axes[1, 0].plot(t_steps, res_ne_tail.state_history[:, 0], color=RIS_GREEN, lw=1.8, label=r"Tailored State $\hat{u}_1(t)$")
    axes[1, 0].plot(t_steps, res_ne_def.state_history[:, 0], color=ORANGE_DEFAULT, lw=1.5, alpha=0.85, label=r"Default State $\hat{u}_1(t)$")
    axes[1, 0].axhline(u_star[0], color=BLACK_NE, linestyle="--", lw=1.5, label=f"True NE $u_1^* = {u_star[0]:.3f}$")
    axes[1, 0].axhline(lb0, color="gray", linestyle=":", lw=1.2, alpha=0.6)
    axes[1, 0].axhline(ub0, color="gray", linestyle=":", lw=1.2, alpha=0.6)
    axes[1, 0].set_title(r"Player 1: Demodulated State History $\hat{u}_1(t)$")
    axes[1, 0].set_xlabel("Iteration Step $t$")
    axes[1, 0].set_ylabel(r"State $\hat{u}_1$")
    axes[1, 0].grid(True, alpha=0.3)
    axes[1, 0].legend(loc="best", fontsize=8.5)

    # (d) Player 2 Demodulated State History
    axes[1, 1].plot(t_steps, res_ne_tail.state_history[:, 1], color=RIS_GREEN, lw=1.8, label=r"Tailored State $\hat{u}_2(t)$")
    axes[1, 1].plot(t_steps, res_ne_def.state_history[:, 1], color=ORANGE_DEFAULT, lw=1.5, alpha=0.85, label=r"Default State $\hat{u}_2(t)$")
    axes[1, 1].axhline(u_star[1], color=BLACK_NE, linestyle="--", lw=1.5, label=f"True NE $u_2^* = {u_star[1]:.3f}$")
    axes[1, 1].axhline(lb1, color="gray", linestyle=":", lw=1.2, alpha=0.6)
    axes[1, 1].axhline(ub1, color="gray", linestyle=":", lw=1.2, alpha=0.6)
    axes[1, 1].set_title(r"Player 2: Demodulated State History $\hat{u}_2(t)$")
    axes[1, 1].set_xlabel("Iteration Step $t$")
    axes[1, 1].set_ylabel(r"State $\hat{u}_2$")
    axes[1, 1].grid(True, alpha=0.3)
    axes[1, 1].legend(loc="best", fontsize=8.5)

    plt.tight_layout()
    fig_path = os.path.join(data_dir, f"rep_game_{gid}_neseeking.png")
    plt.savefig(fig_path, dpi=300, bbox_inches="tight")
    plt.close(fig)

    print(f"Game {gid:2d} ({category[:20]}): Generated 4-panel trajectory plots (Action 1, Action 2, State 1, State 2).")
print(f"\nAll representative game plots saved to: {data_dir}")

## 2. Sobol Global Variance Sensitivity Analysis (Parallel Saltelli Scheme)

To rigorously identify which hyperparameter dimensions drive algorithm performance and identify sources of variance, we perform a **Sobol Variance Decomposition** across the 5 representative games:
- **Sample Size**: $N = 256 = 2^8$ quasi-random base draws using scrambled Sobol sequences (`scipy.stats.qmc.Sobol`), achieving optimal base-2 dyadic grid stratification.
- **Simulation Horizon**: $T = 5,000$ simulation steps per trajectory.
- **Estimators**:
  - **First-Order Index ($S_i$)**: Quantifies main-effects variance (Saltelli, 2010):
    $$S_i = \frac{\frac{1}{N} \sum_{j=1}^N y_B^{(j)} (y_{A_B^{(i)}}^{(j)} - y_A^{(j)})}{\mathbb{V}[y]}$$
  - **Total-Order Index ($S_{Ti}$)**: Captures main effects plus all higher-order cross-interactions (Jansen, 1999):
    $$S_{Ti} = \frac{\frac{1}{2N} \sum_{j=1}^N (y_A^{(j)} - y_{A_B^{(i)}}^{(j)})^2}{\mathbb{V}[y]}$$
- **Parameter Spaces**:
  - **NESeeking5D ($d = 5$ dimensions)**: $k_1, k_2 \in [0.01, 100.0]$, $a_1, a_2 \in [0.005, 2.0]$ (scaled by player box width), $\omega_1 \in [1.0, 50.0]$.
  - **ZOGrandientPlay ($d = 3$ dimensions)**: $a \in [0.005, 2.0]$ (scaled by min box width), $v \in [0.01, 100.0]$, $\epsilon \in [10^{-5}, 0.20]$.

In [ ]:
# Parallel Sobol Global Sensitivity Analysis (N = 256, T = 5,000 steps)
sobol_csv_path = os.path.join(data_dir, "sobol_sensitivity_results.csv")
n_sobol = 256
t_sobol_sim = 5000

if os.path.exists(sobol_csv_path):
    print(f"Loading cached Sobol sensitivity results from: {sobol_csv_path}")
    sobol_df = pd.read_csv(sobol_csv_path)
else:
    print(f"Executing Parallel Sobol Sensitivity Analysis (N = {n_sobol}, 5 Games)...", flush=True)
    t0_sob = time.time()
    sobol_records = []
    rep_ids = [gid for gid, _ in rep_games_info]

    # ZOGP Sobol (d = 3)
    for gid in rep_ids:
        print(f"  Evaluating ZOGP Sobol for Game {gid}...", flush=True)
        s_idx, st_idx = run_game_sobol(games[gid], dim=3, worker_fn=eval_zo_sobol_worker, n_sobol=n_sobol, t_sim=t_sobol_sim, seed=14062003 + gid)
        for p_name, s_val, st_val in zip(["a", "v", "epsilon"], s_idx, st_idx):
            sobol_records.append({"algorithm": "ZOGP", "game_id": gid, "parameter": p_name, "S1": s_val, "ST": st_val})

    # NESeeking Sobol (d = 5)
    for gid in rep_ids:
        print(f"  Evaluating NESeeking5D Sobol for Game {gid}...", flush=True)
        s_idx, st_idx = run_game_sobol(games[gid], dim=5, worker_fn=eval_ne_sobol_worker, n_sobol=n_sobol, t_sim=t_sobol_sim, seed=14062003 + gid)
        for p_name, s_val, st_val in zip(["k1", "k2", "a1", "a2", "omega1"], s_idx, st_idx):
            sobol_records.append({"algorithm": "NESeeking5D", "game_id": gid, "parameter": p_name, "S1": s_val, "ST": st_val})

    sobol_df = pd.DataFrame(sobol_records)
    sobol_df.to_csv(sobol_csv_path, index=False)
    print(f"Sobol analysis completed and saved in {time.time() - t0_sob:.2f}s")

# Visualizing Sobol Indices with Multi-Color Palette (Game 11 in RIS-Grün, distinct colors for others)
for algo, p_labels in [("ZOGP", ["a", "v", "epsilon"]), ("NESeeking5D", ["k1", "k2", "a1", "a2", "omega1"])]:
    df_a = sobol_df[sobol_df["algorithm"] == algo]
    dim = len(p_labels)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))
    fig.suptitle(f"{algo} Global Sensitivity Analysis: Sobol Indices across 5 Representative Games", fontsize=13, fontweight="bold")
    bar_w = 0.15
    x_indices = np.arange(dim)

    for i, (gid, cat) in enumerate(rep_games_info):
        sub = df_a[df_a["game_id"] == gid]
        color = SOBOL_PALETTE[i % len(SOBOL_PALETTE)]
        s1_vals = [sub[sub["parameter"] == p]["S1"].values[0] for p in p_labels]
        st_vals = [sub[sub["parameter"] == p]["ST"].values[0] for p in p_labels]
        ax1.bar(x_indices + i * bar_w, s1_vals, width=bar_w, label=f"Game {gid} ({cat.split('(')[0].strip()})", color=color, alpha=0.85)
        ax2.bar(x_indices + i * bar_w, st_vals, width=bar_w, label=f"Game {gid} ({cat.split('(')[0].strip()})", color=color, alpha=0.85)

    ax1.set_title("First-Order Sobol Index ($)")
    ax1.set_xticks(x_indices + 2 * bar_w)
    ax1.set_xticklabels(p_labels, fontsize=11)
    ax1.grid(True, axis="y", alpha=0.3)
    ax1.legend(loc="upper right", fontsize=8.0)
    ax2.set_title("Total-Order Sobol Index ($)")
    ax2.set_xticks(x_indices + 2 * bar_w)
    ax2.set_xticklabels(p_labels, fontsize=11)
    ax2.grid(True, axis="y", alpha=0.3)
    ax2.axhline(0.05, color="red", linestyle="--", alpha=0.7, label="Significance Threshold (0.05)")
    ax2.legend(loc="upper right", fontsize=8.0)
    plt.tight_layout()
    plot_path = os.path.join(data_dir, f"sobol_sensitivity_{algo.lower()}.png")
    plt.savefig(plot_path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved {algo} Sobol sensitivity plot to: {plot_path}")


## 3. 100-Game Ensemble: Tunability Correlations with Game Geometry

We analyze the relationship between the **Tunability Index** $U(g)$ and the intrinsic geometric characteristics of each game:
$$U(g) = \text{Score}(\theta_{\text{default}}) - \text{Score}(\theta_{\text{tailored}}) = \log_{10}(\text{Regret}_{\text{def}}) - \log_{10}(\text{Regret}_{\text{tailored}})$$

### 4 Defining Geometric Game Characteristics:
1. **Action Set Diameter / Size**: Euclidean span of the compact hyper-rectangle: $\text{diam} = \|\bar{a} - \underline{a}\|_2$.
2. **Condition Number $\kappa$**: Ratio of extreme eigenvalues of the symmetric pseudo-gradient Jacobian: $\kappa = \frac{\lambda_{\max}(-H_{\text{sym}})}{\lambda_{\min}(-H_{\text{sym}})}$.
3. **Boundary Closeness of Nash Equilibrium**: Distance from interior equilibrium to the nearest constraint boundary: $d_{\text{bnd}} = \min_i \min(u_i^* - \underline{a}_i, \bar{a}_i - u_i^*)$.
4. **Coupling Skewness**: Frobenius norm of cross-player interaction asymmetry: $\|H - H^T\|_F$.

In [ ]:
# Compute the 4 defining characteristics for all 100 games
game_records = []
for i, g in enumerate(games):
    lb = np.array([[iv[0] for iv in s] for s in g.action_sets]).ravel()
    ub = np.array([[iv[1] for iv in s] for s in g.action_sets]).ravel()
    u_star = g.nash_equilibrium.ravel()
    diam = float(np.linalg.norm(ub - lb))
    H_sym = 0.5 * (g.H + g.H.T)
    w = np.linalg.eigvalsh(-H_sym)
    kappa = float(w[-1] / max(w[0], 1e-12))
    dist_bnd = float(np.min(np.minimum(u_star - lb, ub - u_star)))
    skew = float(np.linalg.norm(g.H - g.H.T, ord="fro"))
    game_records.append({
        "game_id": i,
        "action_diameter": diam,
        "kappa": kappa,
        "boundary_distance": dist_bnd,
        "skewness": skew,
    })

char_df = pd.DataFrame(game_records)
df_100_all = pd.merge(ensemble_metrics_df, char_df, on="game_id")

char_configs = [
    ("action_diameter", r"Action Set Diameter $\|a_{\max} - a_{\min}\|_2$"),
    ("kappa", r"Condition Number $\kappa$"),
    ("boundary_distance", r"Boundary Distance $\min_i(u_i^* - a_{\min,i}, a_{\max,i} - u_i^*)$"),
    ("skewness", r"Coupling Skewness $\|H - H^T\|_F$"),
]

# -------------------------------------------------------------
# 1. 4 Pairs of Point Plots: Tunability Gain vs. Characteristics
# -------------------------------------------------------------
fig, axes = plt.subplots(4, 2, figsize=(14, 18))
fig.suptitle("100-Game Tunability Gain vs. Geometric Game Characteristics", fontsize=15, fontweight="bold", y=0.995)

for row_idx, (char_col, char_label) in enumerate(char_configs):
    x_vals = df_100_all[char_col].values

    # Left column: ZOGP Tunability
    y_zo = df_100_all["zo_tunability_gain"].values
    r_zo, p_zo = stats.pearsonr(x_vals, y_zo)
    rho_zo, prho_zo = stats.spearmanr(x_vals, y_zo)
    axes[row_idx, 0].scatter(x_vals, y_zo, color=RIS_GREEN, alpha=0.75, edgecolors="none", s=45)
    slope, intercept = np.polyfit(x_vals, y_zo, 1)
    x_grid = np.linspace(min(x_vals), max(x_vals), 100)
    axes[row_idx, 0].plot(x_grid, slope * x_grid + intercept, color="#005948", lw=1.5, linestyle="--")
    axes[row_idx, 0].set_title(f"ZOGP: Tunability vs. {char_label}\nPearson r={r_zo:+.3f} (p={p_zo:.2e}) | Spearman rho={rho_zo:+.3f} (p={prho_zo:.2e})", fontsize=10.5)
    axes[row_idx, 0].set_xlabel(char_label)
    axes[row_idx, 0].set_ylabel("ZOGP Tunability Gain (g)$")
    axes[row_idx, 0].grid(True, alpha=0.3)

    # Right column: NESeeking5D Tunability
    y_ne = df_100_all["ne_tunability_gain"].values
    r_ne, p_ne = stats.pearsonr(x_vals, y_ne)
    rho_ne, prho_ne = stats.spearmanr(x_vals, y_ne)
    axes[row_idx, 1].scatter(x_vals, y_ne, color=RIS_GREEN, alpha=0.75, edgecolors="none", s=45)
    slope_ne, intercept_ne = np.polyfit(x_vals, y_ne, 1)
    axes[row_idx, 1].plot(x_grid, slope_ne * x_grid + intercept_ne, color="#005948", lw=1.5, linestyle="--")
    axes[row_idx, 1].set_title(f"NESeeking5D: Tunability vs. {char_label}\nPearson r={r_ne:+.3f} (p={p_ne:.2e}) | Spearman rho={rho_ne:+.3f} (p={prho_ne:.2e})", fontsize=10.5)
    axes[row_idx, 1].set_xlabel(char_label)
    axes[row_idx, 1].set_ylabel("NESeeking Tunability Gain (g)$")
    axes[row_idx, 1].grid(True, alpha=0.3)

plt.tight_layout()
tunability_plot_path = os.path.join(data_dir, "tunability_vs_characteristics.png")
plt.savefig(tunability_plot_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved 4 pairs of tunability correlation scatter plots to: {tunability_plot_path}")

# ---------------------------------------------------------------------------------
# 2. 4 Pairs of Point Plots: Tailored Total Log-Regret vs. Characteristics (RIS-Grün)
# ---------------------------------------------------------------------------------
fig, axes = plt.subplots(4, 2, figsize=(14, 18))
fig.suptitle("100-Game Tailored Total Log-Regret vs. Geometric Game Characteristics (RIS-Grün)", fontsize=15, fontweight="bold", y=0.995)

for row_idx, (char_col, char_label) in enumerate(char_configs):
    x_vals = df_100_all[char_col].values

    # Left column: ZOGP Tailored Log-Regret
    y_zo = df_100_all["zo_loss_tailored"].values
    r_zo, p_zo = stats.pearsonr(x_vals, y_zo)
    rho_zo, prho_zo = stats.spearmanr(x_vals, y_zo)
    axes[row_idx, 0].scatter(x_vals, y_zo, color=RIS_GREEN, alpha=0.75, edgecolors="none", s=45)
    slope, intercept = np.polyfit(x_vals, y_zo, 1)
    x_grid = np.linspace(min(x_vals), max(x_vals), 100)
    axes[row_idx, 0].plot(x_grid, slope * x_grid + intercept, color="#005948", lw=1.5, linestyle="--")
    axes[row_idx, 0].set_title(f"ZOGP Tailored: Total Log-Regret vs. {char_label}\nPearson r={r_zo:+.3f} (p={p_zo:.2e}) | Spearman rho={rho_zo:+.3f} (p={prho_zo:.2e})", fontsize=10.5)
    axes[row_idx, 0].set_xlabel(char_label)
    axes[row_idx, 0].set_ylabel("ZOGP Tailored Total Log-Regret")
    axes[row_idx, 0].grid(True, alpha=0.3)

    # Right column: NESeeking5D Tailored Log-Regret
    y_ne = df_100_all["ne_loss_tailored"].values
    r_ne, p_ne = stats.pearsonr(x_vals, y_ne)
    rho_ne, prho_ne = stats.spearmanr(x_vals, y_ne)
    axes[row_idx, 1].scatter(x_vals, y_ne, color=RIS_GREEN, alpha=0.75, edgecolors="none", s=45)
    slope_ne, intercept_ne = np.polyfit(x_vals, y_ne, 1)
    axes[row_idx, 1].plot(x_grid, slope_ne * x_grid + intercept_ne, color="#005948", lw=1.5, linestyle="--")
    axes[row_idx, 1].set_title(f"NESeeking5D Tailored: Total Log-Regret vs. {char_label}\nPearson r={r_ne:+.3f} (p={p_ne:.2e}) | Spearman rho={rho_ne:+.3f} (p={prho_ne:.2e})", fontsize=10.5)
    axes[row_idx, 1].set_xlabel(char_label)
    axes[row_idx, 1].set_ylabel("NESeeking Tailored Total Log-Regret")
    axes[row_idx, 1].grid(True, alpha=0.3)

plt.tight_layout()
tailored_regret_plot_path = os.path.join(data_dir, "regret_vs_characteristics_tailored.png")
plt.savefig(tailored_regret_plot_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved 4 pairs of tailored total log-regret scatter plots to: {tailored_regret_plot_path}")

# ---------------------------------------------------------------------------------
# 3. 4 Pairs of Point Plots: Default Total Log-Regret vs. Characteristics (Orange)
# ---------------------------------------------------------------------------------
fig, axes = plt.subplots(4, 2, figsize=(14, 18))
fig.suptitle("100-Game Default Total Log-Regret vs. Geometric Game Characteristics (Orange)", fontsize=15, fontweight="bold", y=0.995)

for row_idx, (char_col, char_label) in enumerate(char_configs):
    x_vals = df_100_all[char_col].values

    # Left column: ZOGP Default Log-Regret
    y_zo = df_100_all["zo_loss_default"].values
    r_zo, p_zo = stats.pearsonr(x_vals, y_zo)
    rho_zo, prho_zo = stats.spearmanr(x_vals, y_zo)
    axes[row_idx, 0].scatter(x_vals, y_zo, color=ORANGE_DEFAULT, alpha=0.75, edgecolors="none", s=45)
    slope, intercept = np.polyfit(x_vals, y_zo, 1)
    x_grid = np.linspace(min(x_vals), max(x_vals), 100)
    axes[row_idx, 0].plot(x_grid, slope * x_grid + intercept, color="#C0392B", lw=1.5, linestyle="--")
    axes[row_idx, 0].set_title(f"ZOGP Default: Total Log-Regret vs. {char_label}\nPearson r={r_zo:+.3f} (p={p_zo:.2e}) | Spearman rho={rho_zo:+.3f} (p={prho_zo:.2e})", fontsize=10.5)
    axes[row_idx, 0].set_xlabel(char_label)
    axes[row_idx, 0].set_ylabel("ZOGP Default Total Log-Regret")
    axes[row_idx, 0].grid(True, alpha=0.3)

    # Right column: NESeeking5D Default Log-Regret
    y_ne = df_100_all["ne_loss_default"].values
    r_ne, p_ne = stats.pearsonr(x_vals, y_ne)
    rho_ne, prho_ne = stats.spearmanr(x_vals, y_ne)
    axes[row_idx, 1].scatter(x_vals, y_ne, color=ORANGE_DEFAULT, alpha=0.75, edgecolors="none", s=45)
    slope_ne, intercept_ne = np.polyfit(x_vals, y_ne, 1)
    axes[row_idx, 1].plot(x_grid, slope_ne * x_grid + intercept_ne, color="#C0392B", lw=1.5, linestyle="--")
    axes[row_idx, 1].set_title(f"NESeeking5D Default: Total Log-Regret vs. {char_label}\nPearson r={r_ne:+.3f} (p={p_ne:.2e}) | Spearman rho={rho_ne:+.3f} (p={prho_ne:.2e})", fontsize=10.5)
    axes[row_idx, 1].set_xlabel(char_label)
    axes[row_idx, 1].set_ylabel("NESeeking Default Total Log-Regret")
    axes[row_idx, 1].grid(True, alpha=0.3)

plt.tight_layout()
default_regret_plot_path = os.path.join(data_dir, "regret_vs_characteristics_default.png")
plt.savefig(default_regret_plot_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved 4 pairs of default total log-regret scatter plots to: {default_regret_plot_path}")

# ------------------------------------------------------------------------------------------
# 4. 4 Pairs of Point Plots: Total Log-Regret Comparison: Tailored (Green) vs. Default (Orange)
# ------------------------------------------------------------------------------------------
fig, axes = plt.subplots(4, 2, figsize=(14, 18))
fig.suptitle("100-Game Total Log-Regret vs. Characteristics: Tailored (RIS-Grün) vs. Default (Orange)", fontsize=15, fontweight="bold", y=0.995)

for row_idx, (char_col, char_label) in enumerate(char_configs):
    x_vals = df_100_all[char_col].values

    # Left column: ZOGP Comparison
    y_zo_tail = df_100_all["zo_loss_tailored"].values
    y_zo_def = df_100_all["zo_loss_default"].values
    axes[row_idx, 0].scatter(x_vals, y_zo_tail, color=RIS_GREEN, alpha=0.75, label="Tailored Optima", s=45)
    axes[row_idx, 0].scatter(x_vals, y_zo_def, color=ORANGE_DEFAULT, alpha=0.60, label="Universal Default", s=40)
    s_t, i_t = np.polyfit(x_vals, y_zo_tail, 1)
    s_d, i_d = np.polyfit(x_vals, y_zo_def, 1)
    x_grid = np.linspace(min(x_vals), max(x_vals), 100)
    axes[row_idx, 0].plot(x_grid, s_t * x_grid + i_t, color="#005948", lw=1.5, linestyle="--", label="Tailored Trend")
    axes[row_idx, 0].plot(x_grid, s_d * x_grid + i_d, color="#C0392B", lw=1.5, linestyle=":", label="Default Trend")
    axes[row_idx, 0].set_title(f"ZOGP: Total Log-Regret vs. {char_label}", fontsize=10.5)
    axes[row_idx, 0].set_xlabel(char_label)
    axes[row_idx, 0].set_ylabel("Total Log-Regret")
    axes[row_idx, 0].grid(True, alpha=0.3)
    axes[row_idx, 0].legend(loc="best", fontsize=8.5)

    # Right column: NESeeking5D Comparison
    y_ne_tail = df_100_all["ne_loss_tailored"].values
    y_ne_def = df_100_all["ne_loss_default"].values
    axes[row_idx, 1].scatter(x_vals, y_ne_tail, color=RIS_GREEN, alpha=0.75, label="Tailored Optima", s=45)
    axes[row_idx, 1].scatter(x_vals, y_ne_def, color=ORANGE_DEFAULT, alpha=0.60, label="Universal Default", s=40)
    s_nt, i_nt = np.polyfit(x_vals, y_ne_tail, 1)
    s_nd, i_nd = np.polyfit(x_vals, y_ne_def, 1)
    axes[row_idx, 1].plot(x_grid, s_nt * x_grid + i_nt, color="#005948", lw=1.5, linestyle="--", label="Tailored Trend")
    axes[row_idx, 1].plot(x_grid, s_nd * x_grid + i_nd, color="#C0392B", lw=1.5, linestyle=":", label="Default Trend")
    axes[row_idx, 1].set_title(f"NESeeking5D: Total Log-Regret vs. {char_label}", fontsize=10.5)
    axes[row_idx, 1].set_xlabel(char_label)
    axes[row_idx, 1].set_ylabel("Total Log-Regret")
    axes[row_idx, 1].grid(True, alpha=0.3)
    axes[row_idx, 1].legend(loc="best", fontsize=8.5)

plt.tight_layout()
comp_plot_path = os.path.join(data_dir, "regret_vs_characteristics_comparison.png")
plt.savefig(comp_plot_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved comparison total log-regret scatter plots to: {comp_plot_path}")


## 4. 100-Game Ensemble: Convergence & Stopping Horizon Probes (10% & 5% Action Diameter Balls)

To probe the intrinsic difficulty of each game and the sensitivity to target tolerance ball size without confounding from tailored hyperparameter adjustments, all 100 games are evaluated under the fixed **Universal Consensus Default Hyperparameters** across two scale-invariant tolerance regimes:
1. **10% Action Set Diameter Ball**:
   $$\|\text{state}_t - u^*\|_2 \le \epsilon_{\text{tol}}(g) = 0.10 \times \text{diam}(\mathcal{A}_g) = 0.10 \times \|a_{\max} - a_{\min}\|_2$$
2. **5% Action Set Diameter Ball**:
   $$\|\text{state}_t - u^*\|_2 \le \epsilon_{\text{tol}}(g) = 0.05 \times \text{diam}(\mathcal{A}_g) = 0.05 \times \|a_{\max} - a_{\min}\|_2$$
- **Maximum Horizon**: $T_{\max} = 1,000,000$ ($10^6$) iterations.
- **Styling**: Point plots are rendered in Orange (`#FF7F0E`) reflecting Universal Default parameters with logarithmic y-scales for steps. Metric distributions are visualized via 4-panel histograms in RIS-Grün (`#009D81`).


In [ ]:
# Parallel Convergence Probes across 100 games: 10% and 5% Action Set Diameter Balls
conv_csv_10_path = os.path.join(data_dir, "convergence_100_games_10pct.csv")
conv_csv_05_path = os.path.join(data_dir, "convergence_100_games_05pct.csv")
combined_csv_path = os.path.join(data_dir, "convergence_100_games_combined.csv")

# 1. 10% Diameter Ball Probe
if os.path.exists(conv_csv_10_path):
    print(f"Loading existing 10% ball dataset from: {conv_csv_10_path}")
    df_10 = pd.read_csv(conv_csv_10_path)
else:
    print("Running Parallel Convergence Probe across 100 games (tol=10% diam, max_steps=1,000,000, 5 seeds ZOGP)...", flush=True)
    t0_conv = time.time()
    tasks_10 = [
        (
            gid,
            games[gid],
            zo_default,
            ne_default,
            0.10 * char_df.loc[char_df["game_id"] == gid, "action_diameter"].values[0],
            1000000,
        )
        for gid in range(100)
    ]
    with ProcessPoolExecutor(max_workers=8) as executor:
        results_10 = list(executor.map(run_convergence_worker, tasks_10, chunksize=4))
    df_10 = pd.DataFrame(results_10)
    df_10.to_csv(conv_csv_10_path, index=False)
    print(f"10% probe completed in {time.time() - t0_conv:.2f}s and saved to: {conv_csv_10_path}")

# 2. 5% Diameter Ball Probe
if os.path.exists(conv_csv_05_path):
    print(f"Loading existing 5% ball dataset from: {conv_csv_05_path}")
    df_05 = pd.read_csv(conv_csv_05_path)
else:
    print("Running Parallel Convergence Probe across 100 games (tol=5% diam, max_steps=1,000,000, 5 seeds ZOGP)...", flush=True)
    t0_conv = time.time()
    tasks_05 = [
        (
            gid,
            games[gid],
            zo_default,
            ne_default,
            0.05 * char_df.loc[char_df["game_id"] == gid, "action_diameter"].values[0],
            1000000,
        )
        for gid in range(100)
    ]
    with ProcessPoolExecutor(max_workers=8) as executor:
        results_05 = list(executor.map(run_convergence_worker, tasks_05, chunksize=4))
    df_05 = pd.DataFrame(results_05)
    df_05.to_csv(conv_csv_05_path, index=False)
    print(f"5% probe completed in {time.time() - t0_conv:.2f}s and saved to: {conv_csv_05_path}")

# 3. Combined Dataset (Additive - No substitution)
df_merged = pd.merge(
    df_10.rename(columns={
        "ne_tol": "ne_tol_10pct",
        "zo_steps": "zo_steps_10pct",
        "zo_log_steps": "zo_log_steps_10pct",
        "zo_log_regret": "zo_log_regret_10pct",
        "zo_converged": "zo_converged_10pct",
        "zo_conv_rate": "zo_conv_rate_10pct",
        "ne_steps": "ne_steps_10pct",
        "ne_log_steps": "ne_log_steps_10pct",
        "ne_log_regret": "ne_log_regret_10pct",
        "ne_converged": "ne_converged_10pct",
    }),
    df_05.rename(columns={
        "ne_tol": "ne_tol_05pct",
        "zo_steps": "zo_steps_05pct",
        "zo_log_steps": "zo_log_steps_05pct",
        "zo_log_regret": "zo_log_regret_05pct",
        "zo_converged": "zo_converged_05pct",
        "zo_conv_rate": "zo_conv_rate_05pct",
        "ne_steps": "ne_steps_05pct",
        "ne_log_steps": "ne_log_steps_05pct",
        "ne_log_regret": "ne_log_regret_05pct",
        "ne_converged": "ne_converged_05pct",
    }),
    on="game_id",
)
conv_combined_df = pd.merge(df_merged, char_df, on="game_id")
conv_combined_df.to_csv(combined_csv_path, index=False)
print(f"Saved consolidated additive dataset to: {combined_csv_path}")

# 4. Side-by-Side Summary Comparison Table
summary_rows = [
    {
        "Tolerance Ball": "10% Action Diameter",
        "Algorithm": "ZOGrandientPlay (Default)",
        "Convergence Rate": f"{np.mean(df_10.zo_converged)*100:.1f}%",
        "Median Steps": float(np.median(df_10.zo_steps)),
        "Mean Steps": float(np.mean(df_10.zo_steps)),
        "Step Variance": float(np.var(df_10.zo_steps, ddof=1)),
        "Mean Expected Total Log-Regret": float(np.mean(df_10.zo_log_regret)),
        "Regret Variance": float(np.var(df_10.zo_log_regret, ddof=1)),
        "Mean Expected Log10 Steps": float(np.mean(df_10.zo_log_steps)),
    },
    {
        "Tolerance Ball": "10% Action Diameter",
        "Algorithm": "NESeeking5D (Default)",
        "Convergence Rate": f"{np.mean(df_10.ne_converged)*100:.1f}%",
        "Median Steps": float(np.median(df_10.ne_steps)),
        "Mean Steps": float(np.mean(df_10.ne_steps)),
        "Step Variance": float(np.var(df_10.ne_steps, ddof=1)),
        "Mean Expected Total Log-Regret": float(np.mean(df_10.ne_log_regret)),
        "Regret Variance": float(np.var(df_10.ne_log_regret, ddof=1)),
        "Mean Expected Log10 Steps": float(np.mean(df_10.ne_log_steps)),
    },
    {
        "Tolerance Ball": "5% Action Diameter",
        "Algorithm": "ZOGrandientPlay (Default)",
        "Convergence Rate": f"{np.mean(df_05.zo_converged)*100:.1f}%",
        "Median Steps": float(np.median(df_05.zo_steps)),
        "Mean Steps": float(np.mean(df_05.zo_steps)),
        "Step Variance": float(np.var(df_05.zo_steps, ddof=1)),
        "Mean Expected Total Log-Regret": float(np.mean(df_05.zo_log_regret)),
        "Regret Variance": float(np.var(df_05.zo_log_regret, ddof=1)),
        "Mean Expected Log10 Steps": float(np.mean(df_05.zo_log_steps)),
    },
    {
        "Tolerance Ball": "5% Action Diameter",
        "Algorithm": "NESeeking5D (Default)",
        "Convergence Rate": f"{np.mean(df_05.ne_converged)*100:.1f}%",
        "Median Steps": float(np.median(df_05.ne_steps)),
        "Mean Steps": float(np.mean(df_05.ne_steps)),
        "Step Variance": float(np.var(df_05.ne_steps, ddof=1)),
        "Mean Expected Total Log-Regret": float(np.mean(df_05.ne_log_regret)),
        "Regret Variance": float(np.var(df_05.ne_log_regret, ddof=1)),
        "Mean Expected Log10 Steps": float(np.mean(df_05.ne_log_steps)),
    },
]
summary_comp_df = pd.DataFrame(summary_rows)
summary_comp_path = os.path.join(data_dir, "convergence_summary_comparison.csv")
summary_comp_df.to_csv(summary_comp_path, index=False)

print("\n=================== CONVERGENCE PROBES COMPARISON (10% vs 5% DIAMETER) ===================")
print(summary_comp_df.to_string(index=False))
print(f"Summary table saved to: {summary_comp_path}")

In [ ]:
# 1. 10% Ball: Total Log-Regret vs. Characteristics (Orange)
fig, axes = plt.subplots(4, 2, figsize=(14, 18))
fig.suptitle("100-Game Total Log-Regret vs. Geometric Characteristics (10% Diameter Ball)", fontsize=15, fontweight="bold", y=0.995)
DARK_RED_TREND = "#C0392B"
DARK_BLUE = "#1B4F72"

for row_idx, (char_col, char_label) in enumerate(char_configs):
    x_vals = conv_combined_df[char_col].values
    y_zo = conv_combined_df["zo_log_regret_10pct"].values
    r_zo, p_zo = stats.pearsonr(x_vals, y_zo)
    rho_zo, prho_zo = stats.spearmanr(x_vals, y_zo)
    axes[row_idx, 0].scatter(x_vals, y_zo, color=ORANGE_DEFAULT, alpha=0.75, edgecolors="none", s=45)
    slope, intercept = np.polyfit(x_vals, y_zo, 1)
    x_grid = np.linspace(min(x_vals), max(x_vals), 100)
    axes[row_idx, 0].plot(x_grid, slope * x_grid + intercept, color=DARK_RED_TREND, lw=1.5, linestyle="--")
    axes[row_idx, 0].set_title(f"ZOGP (10% Diameter Ball): Expected Log-Regret vs. {char_label}\nPearson r={r_zo:+.3f} (p={p_zo:.2e}) | Spearman rho={rho_zo:+.3f} (p={prho_zo:.2e})", fontsize=10.5)
    axes[row_idx, 0].set_xlabel(char_label)
    axes[row_idx, 0].set_ylabel("ZOGP Expected Total Log-Regret")
    axes[row_idx, 0].grid(True, alpha=0.3)

    y_ne = conv_combined_df["ne_log_regret_10pct"].values
    r_ne, p_ne = stats.pearsonr(x_vals, y_ne)
    rho_ne, prho_ne = stats.spearmanr(x_vals, y_ne)
    axes[row_idx, 1].scatter(x_vals, y_ne, color=ORANGE_DEFAULT, alpha=0.75, edgecolors="none", s=45)
    slope_ne, intercept_ne = np.polyfit(x_vals, y_ne, 1)
    axes[row_idx, 1].plot(x_grid, slope_ne * x_grid + intercept_ne, color=DARK_RED_TREND, lw=1.5, linestyle="--")
    axes[row_idx, 1].set_title(f"NESeeking5D (10% Diameter Ball): Log-Regret vs. {char_label}\nPearson r={r_ne:+.3f} (p={p_ne:.2e}) | Spearman rho={rho_ne:+.3f} (p={prho_ne:.2e})", fontsize=10.5)
    axes[row_idx, 1].set_xlabel(char_label)
    axes[row_idx, 1].set_ylabel("NESeeking Total Log-Regret")
    axes[row_idx, 1].grid(True, alpha=0.3)

plt.tight_layout()
p_reg_10 = os.path.join(data_dir, "convergence_correlations_regret.png")
plt.savefig(p_reg_10, dpi=300, bbox_inches="tight")
plt.close(fig)

# 2. 10% Ball: Steps to Converge vs. Characteristics (Orange, Log-Scale Y)
fig, axes = plt.subplots(4, 2, figsize=(14, 18))
fig.suptitle("100-Game Steps to Converge vs. Geometric Characteristics (10% Diameter Ball, Logarithmic Y-Scale)", fontsize=15, fontweight="bold", y=0.995)

for row_idx, (char_col, char_label) in enumerate(char_configs):
    x_vals = conv_combined_df[char_col].values
    y_zo = conv_combined_df["zo_steps_10pct"].values
    log_y_zo = conv_combined_df["zo_log_steps_10pct"].values
    r_zo, p_zo = stats.pearsonr(x_vals, log_y_zo)
    rho_zo, prho_zo = stats.spearmanr(x_vals, y_zo)
    axes[row_idx, 0].scatter(x_vals, y_zo, color=ORANGE_DEFAULT, alpha=0.75, edgecolors="none", s=45)
    axes[row_idx, 0].set_yscale("log")
    slope, intercept = np.polyfit(x_vals, log_y_zo, 1)
    x_grid = np.linspace(min(x_vals), max(x_vals), 100)
    axes[row_idx, 0].plot(x_grid, 10.0 ** (slope * x_grid + intercept), color=DARK_RED_TREND, lw=1.5, linestyle="--")
    axes[row_idx, 0].set_title(f"ZOGP (10% Diameter Ball): Expected Steps vs. {char_label}\nPearson r(log)={r_zo:+.3f} (p={p_zo:.2e}) | Spearman rho={rho_zo:+.3f} (p={prho_zo:.2e})", fontsize=10.5)
    axes[row_idx, 0].set_xlabel(char_label)
    axes[row_idx, 0].set_ylabel("ZOGP Expected Steps to Converge (log scale)")
    axes[row_idx, 0].grid(True, alpha=0.3, which="both")

    y_ne = conv_combined_df["ne_steps_10pct"].values
    log_y_ne = conv_combined_df["ne_log_steps_10pct"].values
    r_ne, p_ne = stats.pearsonr(x_vals, log_y_ne)
    rho_ne, prho_ne = stats.spearmanr(x_vals, y_ne)
    axes[row_idx, 1].scatter(x_vals, y_ne, color=ORANGE_DEFAULT, alpha=0.75, edgecolors="none", s=45)
    axes[row_idx, 1].set_yscale("log")
    slope_ne, intercept_ne = np.polyfit(x_vals, log_y_ne, 1)
    axes[row_idx, 1].plot(x_grid, 10.0 ** (slope_ne * x_grid + intercept_ne), color=DARK_RED_TREND, lw=1.5, linestyle="--")
    axes[row_idx, 1].set_title(f"NESeeking5D (10% Diameter Ball): Steps vs. {char_label}\nPearson r(log)={r_ne:+.3f} (p={p_ne:.2e}) | Spearman rho={rho_ne:+.3f} (p={prho_ne:.2e})", fontsize=10.5)
    axes[row_idx, 1].set_xlabel(char_label)
    axes[row_idx, 1].set_ylabel("NESeeking Steps to Converge (log scale)")
    axes[row_idx, 1].grid(True, alpha=0.3, which="both")

plt.tight_layout()
p_steps_10 = os.path.join(data_dir, "convergence_correlations_steps.png")
plt.savefig(p_steps_10, dpi=300, bbox_inches="tight")
plt.close(fig)

# 3. New Histogram 1: Total Log-Regret (Row 0 = 10% Ball, Row 1 = 5% Ball, Col 0 = ZOGP, Col 1 = NESeeking)
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Convergence Total Log-Regret Distributions across 100 Games", fontsize=15, fontweight="bold", y=0.995)

# Row 0, Col 0: ZOGP 10% Ball
zo_r10 = conv_combined_df["zo_log_regret_10pct"].values
axes[0, 0].hist(zo_r10, bins=20, color=RIS_GREEN, edgecolor="#005948", alpha=0.85)
m_zo10 = float(np.mean(zo_r10))
med_zo10 = float(np.median(zo_r10))
axes[0, 0].axvline(m_zo10, color="#C0392B", linestyle="--", lw=1.8, label=f"Mean Expected: {m_zo10:.2f}")
axes[0, 0].axvline(med_zo10, color=DARK_BLUE, linestyle=":", lw=1.8, label=f"Median Expected: {med_zo10:.2f}")
axes[0, 0].set_title("ZOGP: Expected Total Log-Regret (10% Diameter Ball)", fontsize=11, fontweight="bold")
axes[0, 0].set_xlabel(r"Expected Total Log-Regret $\mathbb{E}[\log_{10}(R)]$")
axes[0, 0].set_ylabel("Frequency (Games)")
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].legend(fontsize=9.5)

# Row 0, Col 1: NESeeking5D 10% Ball
ne_r10 = conv_combined_df["ne_log_regret_10pct"].values
axes[0, 1].hist(ne_r10, bins=20, color=RIS_GREEN, edgecolor="#005948", alpha=0.85)
m_ne10 = float(np.mean(ne_r10))
med_ne10 = float(np.median(ne_r10))
axes[0, 1].axvline(m_ne10, color="#C0392B", linestyle="--", lw=1.8, label=f"Mean: {m_ne10:.2f}")
axes[0, 1].axvline(med_ne10, color=DARK_BLUE, linestyle=":", lw=1.8, label=f"Median: {med_ne10:.2f}")
axes[0, 1].set_title("NESeeking5D: Total Log-Regret (10% Diameter Ball)", fontsize=11, fontweight="bold")
axes[0, 1].set_xlabel(r"Total Log-Regret $\log_{10}(R)$")
axes[0, 1].set_ylabel("Frequency (Games)")
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].legend(fontsize=9.5)

# Row 1, Col 0: ZOGP 5% Ball
zo_r05 = conv_combined_df["zo_log_regret_05pct"].values
axes[1, 0].hist(zo_r05, bins=20, color=RIS_GREEN, edgecolor="#005948", alpha=0.85)
m_zo05 = float(np.mean(zo_r05))
med_zo05 = float(np.median(zo_r05))
axes[1, 0].axvline(m_zo05, color="#C0392B", linestyle="--", lw=1.8, label=f"Mean Expected: {m_zo05:.2f}")
axes[1, 0].axvline(med_zo05, color=DARK_BLUE, linestyle=":", lw=1.8, label=f"Median Expected: {med_zo05:.2f}")
axes[1, 0].set_title("ZOGP: Expected Total Log-Regret (5% Diameter Ball)", fontsize=11, fontweight="bold")
axes[1, 0].set_xlabel(r"Expected Total Log-Regret $\mathbb{E}[\log_{10}(R)]$")
axes[1, 0].set_ylabel("Frequency (Games)")
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].legend(fontsize=9.5)

# Row 1, Col 1: NESeeking5D 5% Ball
ne_r05 = conv_combined_df["ne_log_regret_05pct"].values
axes[1, 1].hist(ne_r05, bins=20, color=RIS_GREEN, edgecolor="#005948", alpha=0.85)
m_ne05 = float(np.mean(ne_r05))
med_ne05 = float(np.median(ne_r05))
axes[1, 1].axvline(m_ne05, color="#C0392B", linestyle="--", lw=1.8, label=f"Mean: {m_ne05:.2f}")
axes[1, 1].axvline(med_ne05, color=DARK_BLUE, linestyle=":", lw=1.8, label=f"Median: {med_ne05:.2f}")
axes[1, 1].set_title("NESeeking5D: Total Log-Regret (5% Diameter Ball)", fontsize=11, fontweight="bold")
axes[1, 1].set_xlabel(r"Total Log-Regret $\log_{10}(R)$")
axes[1, 1].set_ylabel("Frequency (Games)")
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].legend(fontsize=9.5)

plt.tight_layout()
p_hist_regret = os.path.join(data_dir, "convergence_histograms_regret.png")
plt.savefig(p_hist_regret, dpi=300, bbox_inches="tight")
plt.close(fig)

# 4. New Histogram 2: Steps to Converge (Row 0 = 10% Ball, Row 1 = 5% Ball, Col 0 = ZOGP, Col 1 = NESeeking)
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Convergence Steps Distributions across 100 Games", fontsize=15, fontweight="bold", y=0.995)

# Row 0, Col 0: ZOGP 10% Ball
zo_s10 = conv_combined_df["zo_log_steps_10pct"].values
axes[0, 0].hist(zo_s10, bins=20, color=RIS_GREEN, edgecolor="#005948", alpha=0.85)
m_szo10 = float(np.mean(zo_s10))
med_szo10 = float(np.median(zo_s10))
axes[0, 0].axvline(m_szo10, color="#C0392B", linestyle="--", lw=1.8, label=f"Mean Expected Log10: {m_szo10:.2f}")
axes[0, 0].axvline(med_szo10, color=DARK_BLUE, linestyle=":", lw=1.8, label=f"Median Expected Log10: {med_szo10:.2f}")
axes[0, 0].set_title("ZOGP: Steps to Converge (10% Diameter Ball)", fontsize=11, fontweight="bold")
axes[0, 0].set_xlabel(r"Expected $\log_{10}(	ext{Steps to Converge})$")
axes[0, 0].set_ylabel("Frequency (Games)")
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].legend(fontsize=9.5)

# Row 0, Col 1: NESeeking5D 10% Ball
ne_s10 = conv_combined_df["ne_log_steps_10pct"].values
axes[0, 1].hist(ne_s10, bins=20, color=RIS_GREEN, edgecolor="#005948", alpha=0.85)
m_sne10 = float(np.mean(ne_s10))
med_sne10 = float(np.median(ne_s10))
axes[0, 1].axvline(m_sne10, color="#C0392B", linestyle="--", lw=1.8, label=f"Mean Log10: {m_sne10:.2f}")
axes[0, 1].axvline(med_sne10, color=DARK_BLUE, linestyle=":", lw=1.8, label=f"Median Log10: {med_sne10:.2f}")
axes[0, 1].set_title("NESeeking5D: Steps to Converge (10% Diameter Ball)", fontsize=11, fontweight="bold")
axes[0, 1].set_xlabel(r"$\log_{10}(	ext{Steps to Converge})$")
axes[0, 1].set_ylabel("Frequency (Games)")
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].legend(fontsize=9.5)

# Row 1, Col 0: ZOGP 5% Ball
zo_s05 = conv_combined_df["zo_log_steps_05pct"].values
axes[1, 0].hist(zo_s05, bins=20, color=RIS_GREEN, edgecolor="#005948", alpha=0.85)
m_szo05 = float(np.mean(zo_s05))
med_szo05 = float(np.median(zo_s05))
axes[1, 0].axvline(m_szo05, color="#C0392B", linestyle="--", lw=1.8, label=f"Mean Expected Log10: {m_szo05:.2f}")
axes[1, 0].axvline(med_szo05, color=DARK_BLUE, linestyle=":", lw=1.8, label=f"Median Expected Log10: {med_szo05:.2f}")
axes[1, 0].set_title("ZOGP: Steps to Converge (5% Diameter Ball)", fontsize=11, fontweight="bold")
axes[1, 0].set_xlabel(r"Expected $\log_{10}(	ext{Steps to Converge})$")
axes[1, 0].set_ylabel("Frequency (Games)")
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].legend(fontsize=9.5)

# Row 1, Col 1: NESeeking5D 5% Ball
ne_s05 = conv_combined_df["ne_log_steps_05pct"].values
axes[1, 1].hist(ne_s05, bins=20, color=RIS_GREEN, edgecolor="#005948", alpha=0.85)
m_sne05 = float(np.mean(ne_s05))
med_sne05 = float(np.median(ne_s05))
axes[1, 1].axvline(m_sne05, color="#C0392B", linestyle="--", lw=1.8, label=f"Mean Log10: {m_sne05:.2f}")
axes[1, 1].axvline(med_sne05, color=DARK_BLUE, linestyle=":", lw=1.8, label=f"Median Log10: {med_sne05:.2f}")
axes[1, 1].set_title("NESeeking5D: Steps to Converge (5% Diameter Ball)", fontsize=11, fontweight="bold")
axes[1, 1].set_xlabel(r"$\log_{10}(	ext{Steps to Converge})$")
axes[1, 1].set_ylabel("Frequency (Games)")
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].legend(fontsize=9.5)

plt.tight_layout()
p_hist_steps = os.path.join(data_dir, "convergence_histograms_steps.png")
plt.savefig(p_hist_steps, dpi=300, bbox_inches="tight")
plt.close(fig)

# 5. 5% Ball: Total Log-Regret vs. Characteristics (Orange)
fig, axes = plt.subplots(4, 2, figsize=(14, 18))
fig.suptitle("100-Game Total Log-Regret vs. Geometric Characteristics (5% Diameter Ball)", fontsize=15, fontweight="bold", y=0.995)

for row_idx, (char_col, char_label) in enumerate(char_configs):
    x_vals = conv_combined_df[char_col].values
    y_zo = conv_combined_df["zo_log_regret_05pct"].values
    r_zo, p_zo = stats.pearsonr(x_vals, y_zo)
    rho_zo, prho_zo = stats.spearmanr(x_vals, y_zo)
    axes[row_idx, 0].scatter(x_vals, y_zo, color=ORANGE_DEFAULT, alpha=0.75, edgecolors="none", s=45)
    slope, intercept = np.polyfit(x_vals, y_zo, 1)
    x_grid = np.linspace(min(x_vals), max(x_vals), 100)
    axes[row_idx, 0].plot(x_grid, slope * x_grid + intercept, color=DARK_RED_TREND, lw=1.5, linestyle="--")
    axes[row_idx, 0].set_title(f"ZOGP (5% Ball): Expected Log-Regret vs. {char_label}\nPearson r={r_zo:+.3f} (p={p_zo:.2e}) | Spearman rho={rho_zo:+.3f} (p={prho_zo:.2e})", fontsize=10.5)
    axes[row_idx, 0].set_xlabel(char_label)
    axes[row_idx, 0].set_ylabel("ZOGP Expected Total Log-Regret")
    axes[row_idx, 0].grid(True, alpha=0.3)

    y_ne = conv_combined_df["ne_log_regret_05pct"].values
    r_ne, p_ne = stats.pearsonr(x_vals, y_ne)
    rho_ne, prho_ne = stats.spearmanr(x_vals, y_ne)
    axes[row_idx, 1].scatter(x_vals, y_ne, color=ORANGE_DEFAULT, alpha=0.75, edgecolors="none", s=45)
    slope_ne, intercept_ne = np.polyfit(x_vals, y_ne, 1)
    axes[row_idx, 1].plot(x_grid, slope_ne * x_grid + intercept_ne, color=DARK_RED_TREND, lw=1.5, linestyle="--")
    axes[row_idx, 1].set_title(f"NESeeking5D (5% Ball): Log-Regret vs. {char_label}\nPearson r={r_ne:+.3f} (p={p_ne:.2e}) | Spearman rho={rho_ne:+.3f} (p={prho_ne:.2e})", fontsize=10.5)
    axes[row_idx, 1].set_xlabel(char_label)
    axes[row_idx, 1].set_ylabel("NESeeking Total Log-Regret")
    axes[row_idx, 1].grid(True, alpha=0.3)

plt.tight_layout()
p_reg_05 = os.path.join(data_dir, "convergence_correlations_regret_05pct.png")
plt.savefig(p_reg_05, dpi=300, bbox_inches="tight")
plt.close(fig)

# 6. 5% Ball: Steps to Converge vs. Characteristics (Orange, Log-Scale Y)
fig, axes = plt.subplots(4, 2, figsize=(14, 18))
fig.suptitle("100-Game Steps to Converge vs. Geometric Characteristics (5% Diameter Ball, Logarithmic Y-Scale)", fontsize=15, fontweight="bold", y=0.995)

for row_idx, (char_col, char_label) in enumerate(char_configs):
    x_vals = conv_combined_df[char_col].values
    y_zo = conv_combined_df["zo_steps_05pct"].values
    log_y_zo = conv_combined_df["zo_log_steps_05pct"].values
    r_zo, p_zo = stats.pearsonr(x_vals, log_y_zo)
    rho_zo, prho_zo = stats.spearmanr(x_vals, y_zo)
    axes[row_idx, 0].scatter(x_vals, y_zo, color=ORANGE_DEFAULT, alpha=0.75, edgecolors="none", s=45)
    axes[row_idx, 0].set_yscale("log")
    slope, intercept = np.polyfit(x_vals, log_y_zo, 1)
    x_grid = np.linspace(min(x_vals), max(x_vals), 100)
    axes[row_idx, 0].plot(x_grid, 10.0 ** (slope * x_grid + intercept), color=DARK_RED_TREND, lw=1.5, linestyle="--")
    axes[row_idx, 0].set_title(f"ZOGP (5% Ball): Expected Steps vs. {char_label}\nPearson r(log)={r_zo:+.3f} (p={p_zo:.2e}) | Spearman rho={rho_zo:+.3f} (p={prho_zo:.2e})", fontsize=10.5)
    axes[row_idx, 0].set_xlabel(char_label)
    axes[row_idx, 0].set_ylabel("ZOGP Expected Steps to Converge (log scale)")
    axes[row_idx, 0].grid(True, alpha=0.3, which="both")

    y_ne = conv_combined_df["ne_steps_05pct"].values
    log_y_ne = conv_combined_df["ne_log_steps_05pct"].values
    r_ne, p_ne = stats.pearsonr(x_vals, log_y_ne)
    rho_ne, prho_ne = stats.spearmanr(x_vals, y_ne)
    axes[row_idx, 1].scatter(x_vals, y_ne, color=ORANGE_DEFAULT, alpha=0.75, edgecolors="none", s=45)
    axes[row_idx, 1].set_yscale("log")
    slope_ne, intercept_ne = np.polyfit(x_vals, log_y_ne, 1)
    axes[row_idx, 1].plot(x_grid, 10.0 ** (slope_ne * x_grid + intercept_ne), color=DARK_RED_TREND, lw=1.5, linestyle="--")
    axes[row_idx, 1].set_title(f"NESeeking5D (5% Ball): Steps vs. {char_label}\nPearson r(log)={r_ne:+.3f} (p={p_ne:.2e}) | Spearman rho={rho_ne:+.3f} (p={prho_ne:.2e})", fontsize=10.5)
    axes[row_idx, 1].set_xlabel(char_label)
    axes[row_idx, 1].set_ylabel("NESeeking Steps to Converge (log scale)")
    axes[row_idx, 1].grid(True, alpha=0.3, which="both")

plt.tight_layout()
p_steps_05 = os.path.join(data_dir, "convergence_correlations_steps_05pct.png")
plt.savefig(p_steps_05, dpi=300, bbox_inches="tight")
plt.close(fig)

# 7. Direct Comparative Plot: 10% vs 5% Ball (2x2 Grid)
fig, axes = plt.subplots(2, 2, figsize=(14, 11))
fig.suptitle("Convergence Sensitivity to Tolerance Ball Size: 10% vs. 5% Action Set Diameter", fontsize=15, fontweight="bold", y=0.995)

axes[0, 0].scatter(conv_combined_df["zo_steps_10pct"], conv_combined_df["zo_steps_05pct"], color=ORANGE_DEFAULT, alpha=0.75, edgecolors="#C0392B", s=50)
axes[0, 0].set_xscale("log")
axes[0, 0].set_yscale("log")
axes[0, 0].plot([1, 1e6], [1, 1e6], color="#333333", linestyle="--", lw=1.5, label="y = x (Identical Steps)")
axes[0, 0].set_title("ZOGP: Expected Steps to Converge (10% vs. 5% Ball)", fontsize=11, fontweight="bold")
axes[0, 0].set_xlabel("Expected Steps to 10% Ball (log scale)")
axes[0, 0].set_ylabel("Expected Steps to 5% Ball (log scale)")
axes[0, 0].grid(True, alpha=0.3, which="both")
axes[0, 0].legend(fontsize=9.5)

axes[0, 1].scatter(conv_combined_df["ne_steps_10pct"], conv_combined_df["ne_steps_05pct"], color=ORANGE_DEFAULT, alpha=0.75, edgecolors="#C0392B", s=50)
axes[0, 1].set_xscale("log")
axes[0, 1].set_yscale("log")
axes[0, 1].plot([100, 1e6], [100, 1e6], color="#333333", linestyle="--", lw=1.5, label="y = x (Identical Steps)")
axes[0, 1].set_title("NESeeking5D: Steps to Converge (10% vs. 5% Ball)", fontsize=11, fontweight="bold")
axes[0, 1].set_xlabel("Steps to 10% Ball (log scale)")
axes[0, 1].set_ylabel("Steps to 5% Ball (log scale)")
axes[0, 1].grid(True, alpha=0.3, which="both")
axes[0, 1].legend(fontsize=9.5)

box_data_zo = [conv_combined_df["zo_log_regret_10pct"].values, conv_combined_df["zo_log_regret_05pct"].values]
bplot_zo = axes[1, 0].boxplot(box_data_zo, tick_labels=["10% Ball", "5% Ball"], patch_artist=True)
for patch in bplot_zo["boxes"]:
    patch.set_facecolor(RIS_GREEN)
    patch.set_alpha(0.7)
axes[1, 0].set_title("ZOGP: Expected Total Log-Regret (10% vs. 5% Ball)", fontsize=11, fontweight="bold")
axes[1, 0].set_ylabel("Expected Total Log-Regret")
axes[1, 0].grid(True, alpha=0.3)

box_data_ne = [conv_combined_df["ne_log_regret_10pct"].values, conv_combined_df["ne_log_regret_05pct"].values]
bplot_ne = axes[1, 1].boxplot(box_data_ne, tick_labels=["10% Ball", "5% Ball"], patch_artist=True)
for patch in bplot_ne["boxes"]:
    patch.set_facecolor(RIS_GREEN)
    patch.set_alpha(0.7)
axes[1, 1].set_title("NESeeking5D: Total Log-Regret (10% vs. 5% Ball)", fontsize=11, fontweight="bold")
axes[1, 1].set_ylabel("Total Log-Regret")
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
p_comp = os.path.join(data_dir, "convergence_comparison_10pct_vs_05pct.png")
plt.savefig(p_comp, dpi=300, bbox_inches="tight")
plt.close(fig)
print("Saved all updated figures successfully.")

## 5. Synthesis & Key Thesis Takeaways

### 1. The Impact of Game Geometry on Algorithm Tunability
- **Boundary Closeness ($d_{\text{bnd}}$)**:
  For `NESeeking5D`, boundary closeness exhibits a statistically significant negative rank correlation with tunability gain (Spearman $\rho = -0.340$, $p = 5.4 \times 10^{-4}$). When the true Nash equilibrium lies near the boundary manifold, default sinusoidal perturbations cause severe boundary projection clipping. Tailored optimization reduces the perturbation amplitude $a_i$ and rebalances integrator gains $k_i$, yielding massive performance recovery on boundary-adjacent games.
- **Coupling Skewness ($\|H - H^T\|_F$)**:
  Higher cross-player coupling asymmetry induces rotational dynamics in pseudo-gradient vector fields, which impacts gradient play and extremum seeking differently. Extremum seeking requires higher dither frequencies to track asymmetric rotational flows without limit-cycle drift.

### 2. Convergence Dynamics & The Role of Default Hyperparameters
- Holding hyperparameters fixed at their **Universal Consensus Default** treats each algorithm as a calibrated measurement instrument, isolating the pure causal effect of game geometry.
- Due to the persistent sinusoidal dither of extremum seeking and stochastic variance of zeroth-order gradient play, algorithms converge to an $\mathcal{O}(a^2)$ or $\mathcal{O}(\sigma^2)$ neighborhood rather than a single point. Specifying $\epsilon_{\text{tol}} = 0.05$ effectively separates games where default parameters achieve steady-state stabilization from those requiring task-specific adaptation.

### 3. Global Parameter Sensitivity (Sobol Decomposition)
- Across the 5 representative games, the total-order Sobol index ($S_T$) reveals that perturbation amplitude ($a$) and integrator gains ($k_i$, $v$) dominate total regret variance, whereas auxiliary decay parameters contribute minimal isolated variance.
- This mathematically justifies reducing the search space from 8D to 5D in `NESeeking` without performance degradation, drastically accelerating multi-task Bayesian optimization.